<a href="https://colab.research.google.com/github/anactechn/CadastroPacientesBasico/blob/main/CadastroPacientes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import csv
import os

def carregar_registros_pacientes():
    """Carrega os registros de pacientes de um arquivo CSV."""
    try:
        with open("registros_pacientes.csv", "r", encoding='utf-8') as arquivo:
            leitor = csv.DictReader(arquivo)
            return list(leitor)
    except FileNotFoundError:
        return []
    except Exception as e:
        print(f"⚠️ Erro ao carregar arquivo: {e}")
        return []

def salvar_registros_pacientes(registros):
    """Salva os registros de pacientes em um arquivo CSV."""
    try:
        if not registros:
            print("⚠️ Nenhum registro para salvar!")
            return

        with open("registros_pacientes.csv", "w", newline="", encoding='utf-8') as arquivo:
            campos = registros[0].keys()
            escritor = csv.DictWriter(arquivo, fieldnames=campos)
            escritor.writeheader()
            escritor.writerows(registros)
        print("✅ Registros salvos com sucesso!")
    except Exception as e:
        print(f"⚠️ Erro ao salvar arquivo: {e}")

def criar_prontuario(registros):
    """Cria um novo prontuário de paciente."""
    print("\n📋 NOVO PRONTUÁRIO")
    print("═"*40)

    paciente = {
        'Nome': input("Nome completo: ").strip(),
        'RG': validar_input("RG (apenas números): ", numerico=True),
        'CPF': validar_input("CPF (apenas números): ", numerico=True, tamanho=11),
        'CNS': validar_input("Cartão Nacional de Saúde (CNS): ", numerico=True),
        'Localizacao': input("Local que se encontra: ").strip(),
        'Contato': validar_input("Telefone para contato (com DDD): ", numerico=True),
        'Unidade': input("Unidade de saúde de referência: ").strip(),
        'Data': validar_data("Data do atendimento (DD/MM/AAAA): "),
        'Problema_Principal': input("Problema de saúde principal: ").strip(),
        'Problema_Secundario': input("Problema de saúde secundário: ").strip(),
        'Tabagismo': validar_sim_nao("Tabagista? (S/N): "),
        'Etilismo': validar_sim_nao("Consumo de álcool? (S/N): "),
        'Medicacao_Principal': input("Medicação principal: ").strip(),
        'Outras_Medicacoes': input("Outras medicações (separar por vírgula): ").strip(),
        'Problema_Odontologico': input("Problema odontológico/recomendações: ").strip(),
        'Pressao_Arterial': validar_pressao("Pressão arterial (ex: 120/80): "),
        'Frequencia_Cardiaca': validar_input("Frequência cardíaca (bpm): ", numerico=True),
        'Saturacao_O2': validar_input("Saturação O₂ (%): ", numerico=True, min_val=0, max_val=100),
        'Glicemia': validar_input("Glicemia capilar (mg/dL): ", numerico=True),
        'Conduta_Medica': input("Conduta médica: ").strip(),
        'Medico_Responsavel': input("Nome do médico responsável: ").strip(),
        'Demanda_Principal': input("Demanda principal: ").strip(),
        'Outras_Demandas': input("Outras demandas (separar por vírgula): ").strip()
    }

    registros.append(paciente)
    print("\n═"*40)
    print("✅ PRONTUÁRIO CRIADO COM SUCESSO!")
    return registros

def validar_input(mensagem, numerico=False, tamanho=None, min_val=None, max_val=None):
    """Valida entradas do usuário."""
    while True:
        try:
            valor = input(mensagem).strip()
            if not valor:
                raise ValueError("Campo obrigatório!")

            if numerico:
                if not valor.isdigit():
                    raise ValueError("Apenas números são permitidos!")
                valor = int(valor)

                if tamanho and len(str(valor)) != tamanho:
                    raise ValueError(f"Deve conter exatamente {tamanho} dígitos!")

                if min_val is not None and valor < min_val:
                    raise ValueError(f"Valor mínimo permitido: {min_val}")

                if max_val is not None and valor > max_val:
                    raise ValueError(f"Valor máximo permitido: {max_val}")

            return str(valor)
        except ValueError as e:
            print(f"⚠️ Erro: {e}")

def validar_sim_nao(mensagem):
    """Valida respostas Sim/Não."""
    while True:
        resposta = input(mensagem).strip().upper()
        if resposta in ('S', 'N'):
            return resposta
        print("⚠️ Por favor, digite S para Sim ou N para Não")

def validar_data(mensagem):
    """Valida a data no formato DD/MM/AAAA."""
    while True:
        data = input(mensagem).strip()
        try:
            dia, mes, ano = map(int, data.split('/'))
            if len(data) != 10 or data[2] != '/' or data[5] != '/':
                raise ValueError
            if not (1 <= dia <= 31 and 1 <= mes <= 12 and 1900 <= ano <= 2100):
                raise ValueError
            return data
        except ValueError:
            print("⚠️ Formato inválido! Use DD/MM/AAAA com valores válidos")

def validar_pressao(mensagem):
    """Valida a pressão arterial no formato XXX/XX."""
    while True:
        pressao = input(mensagem).strip()
        try:
            sistolica, diastolica = pressao.split('/')
            if not (sistolica.isdigit() and diastolica.isdigit()):
                raise ValueError
            if len(sistolica) not in (2,3) or len(diastolica) not in (2,3):
                raise ValueError
            return pressao
        except ValueError:
            print("⚠️ Formato inválido! Use valores como 120/80")

def visualizar_prontuarios(registros):
    """Exibe todos os prontuários cadastrados."""
    print("\n📂 PRONTUÁRIOS CADASTRADOS")
    print("═"*40)

    if not registros:
        print("Nenhum prontuário cadastrado.")
        return

    for i, paciente in enumerate(registros, 1):
        print(f"\n📄 PRONTUÁRIO {i}")
        print("-"*30)
        for chave, valor in paciente.items():
            print(f"{chave.replace('_', ' ').title()}: {valor}")

def editar_prontuario(registros):
    """Permite editar um prontuário existente."""
    if not registros:
        print("Nenhum prontuário cadastrado para editar.")
        return registros

    visualizar_prontuarios(registros)

    try:
        indice = int(input("\nDigite o número do prontuário que deseja editar: ")) - 1
        if indice < 0 or indice >= len(registros):
            raise ValueError

        paciente = registros[indice]
        print("\nDeixe em branco para manter o valor atual.")

        for campo in paciente:
            novo_valor = input(f"{campo.replace('_', ' ').title()} (atual: {paciente[campo]}): ").strip()
            if novo_valor:
                paciente[campo] = novo_valor

        print("\n✅ PRONTUÁRIO ATUALIZADO COM SUCESSO!")
        return registros
    except (ValueError, IndexError):
        print("⚠️ Número de prontuário inválido!")
        return registros

def buscar_prontuario(registros):
    """Busca prontuários por diversos critérios."""
    if not registros:
        print("Nenhum prontuário cadastrado para buscar.")
        return

    print("\n🔍 BUSCAR PRONTUÁRIO")
    print("1. Por nome")
    print("2. Por CPF")
    print("3. Por data de atendimento")
    print("4. Por unidade de saúde")
    opcao = input("Escolha o critério de busca: ")

    resultados = []
    termo = input("Digite o termo de busca: ").strip().lower()

    if opcao == "1":
        resultados = [p for p in registros if termo in p['Nome'].lower()]
    elif opcao == "2":
        resultados = [p for p in registros if termo in p['CPF']]
    elif opcao == "3":
        resultados = [p for p in registros if termo in p['Data']]
    elif opcao == "4":
        resultados = [p for p in registros if termo in p['Unidade'].lower()]
    else:
        print("⚠️ Opção inválida!")
        return

    if resultados:
        print(f"\n🔎 {len(resultados)} PRONTUÁRIO(S) ENCONTRADO(S):")
        visualizar_prontuarios(resultados)
    else:
        print("Nenhum prontuário encontrado com os critérios informados.")

def backup_registros(registros):
    """Cria um backup dos registros."""
    if not registros:
        print("Nenhum registro para fazer backup.")
        return

    import datetime
    data_atual = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    nome_arquivo = f"backup_registros_{data_atual}.csv"

    try:
        with open(nome_arquivo, "w", newline="", encoding='utf-8') as arquivo:
            campos = registros[0].keys()
            escritor = csv.DictWriter(arquivo, fieldnames=campos)
            escritor.writeheader()
            escritor.writerows(registros)
        print(f"✅ Backup criado com sucesso: {nome_arquivo}")
    except Exception as e:
        print(f"⚠️ Erro ao criar backup: {e}")

def main():
    """Função principal do sistema de prontuários."""
    registros = carregar_registros_pacientes()

    while True:
        print("\n🏥 SISTEMA DE PRONTUÁRIOS MÉDICOS")
        print("═"*40)
        print("\nMENU PRINCIPAL:")
        print("1. Criar novo prontuário")
        print("2. Visualizar todos os prontuários")
        print("3. Editar prontuário existente")
        print("4. Buscar prontuários")
        print("5. Criar backup dos registros")
        print("6. Salvar e sair do sistema")

        opcao = input("\nEscolha uma opção: ").strip()

        if opcao == "1":
            registros = criar_prontuario(registros)
        elif opcao == "2":
            visualizar_prontuarios(registros)
        elif opcao == "3":
            registros = editar_prontuario(registros)
        elif opcao == "4":
            buscar_prontuario(registros)
        elif opcao == "5":
            backup_registros(registros)
        elif opcao == "6":
            salvar_registros_pacientes(registros)
            print("\n🚪 Saindo do sistema... Até logo!")
            break
        else:
            print("\n⚠️ Opção inválida! Por favor, escolha uma opção de 1 a 6.")

if __name__ == "__main__":
    main()


🏥 SISTEMA DE PRONTUÁRIOS MÉDICOS
════════════════════════════════════════

MENU PRINCIPAL:
1. Criar novo prontuário
2. Visualizar todos os prontuários
3. Editar prontuário existente
4. Buscar prontuários
5. Criar backup dos registros
6. Salvar e sair do sistema


KeyboardInterrupt: Interrupted by user